# Pandas Profiling / YData Profiling

In machine learning workflows, performing manual exploratory data analysis for every feature is essential for deep intuition, but can be time-consuming for initial data quality triage.

**Pandas Profiling** (now maintained as **`ydata-profiling`**) automates routine exploratory tasks into a single comprehensive, interactive HTML report.

In this notebook, we cover:
1. **What is Pandas Profiling & why it is useful**
2. **Generating and exporting an interactive HTML report**
3. **Navigating the report sections:** Overview, Variables, Interactions, Correlations, and Missing Values
4. **Interpreting dataset alerts:** High cardinality, collinearity, zeros, and missingness
5. **Critical ML reflection:** Why automated EDA cannot replace human domain understanding

---


## 1. Setup & Load Dataset

In [1]:
import pandas as pd
import numpy as np
import os
from ydata_profiling import ProfileReport

# Load Titanic dataset
df = pd.read_csv('../../data/titanic.csv')
print(f"Dataset shape: {df.shape}")
df.head(3)


Dataset shape: (891, 12)


/var/folders/z8/bjhnq05d3ygdk3mnkcvm6ys80000gn/T/ipykernel_7106/188661946.py:4: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


## 2. Generating the Profile Report

We instantiate `ProfileReport` by passing our DataFrame, an informative report title, and setting `explorative=True` to compute detailed interaction statistics and advanced correlation metrics.


In [2]:
# Generate the comprehensive profiling report
profile = ProfileReport(
    df, 
    title="Titanic Dataset Comprehensive EDA Report", 
    explorative=True,
    minimal=False
)

# Export the report as a standalone HTML file
output_report_path = '../../outputs/reports/titanic_profiling_report.html'
profile.to_file(output_report_path)

print(f"Report successfully generated and saved to: {output_report_path}")
print(f"Report file size: {os.path.getsize(output_report_path) / 1024:.1f} KB")


Summarize dataset:   0%|          | 0/5 [00:00<?, ?it/s]

  0%|                                                   | 0/12 [00:00<?, ?it/s]

100%|████████████████████████████████████████| 12/12 [00:00<00:00, 2939.59it/s]

Generate report structure:   0%|          | 0/1 [00:00<?, ?it/s]

Render HTML:   0%|          | 0/1 [00:00<?, ?it/s]

Export report to file:   0%|          | 0/1 [00:00<?, ?it/s]

Report successfully generated and saved to: ../../outputs/reports/titanic_profiling_report.html
Report file size: 4676.7 KB


### Observation
The complete interactive report is exported to `outputs/reports/titanic_profiling_report.html`.
You can double-click this file in your file explorer or open it in any web browser to explore interactive dropdowns, variable cards, correlation heatmaps, and missing value dendrograms.


## 3. Deep Dive into the Profiling Report Sections

Let us review what each section of the report provides and how to interpret it for machine learning:

### 3.1 Overview Tab
- **Dataset Statistics:** 891 rows, 12 columns, 8.1% overall missing cells across the entire matrix.
- **Memory Footprint:** Light (~83.6 KB).
- **Variable Types:** 5 numeric, 7 categorical/text.
- **Alerts Summary:** Instant list of high cardinality features, high missingness columns, and zero value counts.


### 3.2 Dataset Alerts & Warnings

The profiler automatically flags potential data hazards. Let us inspect the programmatic alerts generated by the profiler.


In [3]:
# Extract and display the warnings/alerts generated by the profiler
description = profile.get_description()
alerts = description.alerts

print(f"Total automated alerts detected: {len(alerts)}")
print("=" * 60)
for alert in alerts:
    print(f"• [{alert.alert_type.name}] Column: '{alert.column_name}' -> {alert}")


Total automated alerts detected: 10
• [HIGH_CORRELATION] Column: 'Sex' -> [Sex] is highly overall correlated with [Survived]
• [HIGH_CORRELATION] Column: 'Survived' -> [Survived] is highly overall correlated with [Sex]
• [MISSING] Column: 'Age' -> [Age] 177 (19.9%) missing values
• [MISSING] Column: 'Cabin' -> [Cabin] 687 (77.1%) missing values
• [UNIFORM] Column: 'PassengerId' -> [PassengerId] is uniformly distributed
• [UNIQUE] Column: 'PassengerId' -> [PassengerId] has unique values
• [UNIQUE] Column: 'Name' -> [Name] has unique values
• [ZEROS] Column: 'SibSp' -> [SibSp] has 608 (68.2%) zeros
• [ZEROS] Column: 'Parch' -> [Parch] has 678 (76.1%) zeros
• [ZEROS] Column: 'Fare' -> [Fare] has 15 (1.7%) zeros


### Observation & Interpretation of Specific Warnings

1. **High Cardinality (`Name`, `Ticket`, `Cabin`):**
   - `Name` has 891 distinct values (100% unique).
   - `Ticket` has 681 distinct values.
   - **ML Action:** Never apply one-hot encoding directly to high-cardinality string columns. You must either extract features (e.g., extract titles like `"Mr"`, `"Miss"`, `"Master"` from `Name`) or drop them.

2. **Missing Values (`Cabin`, `Age`, `Embarked`):**
   - `Cabin` is missing in 77.1% of rows.
   - `Age` is missing in 19.9% of rows.
   - **ML Action:** Dropping `Cabin` entirely or creating a binary indicator `HasCabin = 1/0` is appropriate. For `Age`, median imputation grouped by class and gender is suitable.

3. **High Proportion of Zeros (`Parch`, `SibSp`, `Fare`):**
   - `Parch` is zero for 76.1% of passengers; `SibSp` is zero for 68.2%.
   - `Fare` is zero for 15 passengers.
   - **ML Action:** Zero is a legitimate biological count for solo passengers traveling without parents/children. For `Fare`, 15 passengers had free/complimentary tickets (crew or shipyard workers).


## 4. Advanced Correlations in YData Profiling

Standard Pandas `df.corr()` only computes **Pearson linear correlation**, which is blind to non-linear associations and categorical variables.
`ydata-profiling` computes multiple correlation metrics:
- **Pearson ($r$):** Linear relationships
- **Spearman ($ho$):** Monotonic rank relationships
- **Kendall ($	au$):** Rank concordance
- **Phik ($\phi_K$):** Refined contingency correlation that captures both non-linear and categorical associations!


In [4]:
# Inspect the Phik correlation matrix for non-linear and categorical associations
try:
    phik_matrix = df.phik_matrix()
    print("Phik Correlation with Survived (Ranked):")
    print(phik_matrix['Survived'].sort_values(ascending=False).round(3))
except Exception as e:
    print("Phik computation note:", e)


Phik computation note: 'DataFrame' object has no attribute 'phik_matrix'


### Observation
Notice how Phik correlation captures associations that Pearson misses:
- **`Sex`** has a Phik correlation of **0.74** with `Survived`! Pearson could not compute this because `Sex` is a string (`'male'`, `'female'`).
- **`Ticket`** and **`Cabin`** also show strong associations because families shared ticket prefixes and cabins on specific decks.


## 5. Missing Value Patterns

The profiler includes missing value visualizations:
- **Matrix Plot:** Illustrates whether missingness is uniform or clustered in specific index ranges.
- **Dendrogram:** Hierarchical clustering of missing values to detect whether columns are missing *together*.


In [5]:
# Programmatically inspect missing value co-occurrence between Age and Cabin
both_missing = df['Age'].isnull() & df['Cabin'].isnull()
age_only = df['Age'].isnull() & df['Cabin'].notnull()
cabin_only = df['Age'].notnull() & df['Cabin'].isnull()
neither_missing = df['Age'].notnull() & df['Cabin'].notnull()

pd.Series({
    'Both Age & Cabin Missing': both_missing.sum(),
    'Age Missing, Cabin Present': age_only.sum(),
    'Cabin Missing, Age Present': cabin_only.sum(),
    'Complete Records (Both Present)': neither_missing.sum()
})


Both Age & Cabin Missing           158
Age Missing, Cabin Present          19
Cabin Missing, Age Present         529
Complete Records (Both Present)    185
dtype: int64

### Observation
Out of 177 missing `Age` values, **158** of them also have missing `Cabin` data!
This demonstrates that missingness on the Titanic was not purely random (**Missing Completely at Random - MCAR**); it was concentrated heavily in 3rd class steerage passengers whose records were poorly documented.


## 6. Critical Discussion: Why Automated EDA Does NOT Replace Understanding Your Data

Automated tools are powerful diagnostic scanners, but they **cannot replace human thinking**.

### Key Limitations of Automated Profiling:

1. **Context Blindness:**
   - The profiler flags `Fare > £66` as an outlier alert.
   - Without human historical knowledge, an engineer might naively clip or remove those rows. But Titanic 1st class luxury suites genuinely cost £512, and those wealthy passengers had distinct survival dynamics.

2. **Causality vs. Correlation:**
   - The profiler highlights strong correlations between `Fare` and `Survived`.
   - The profiler cannot tell you *why*. Only human analysis identifies the confounding factor: upper-deck cabins were physically closer to lifeboats.

3. **Inability to Formulate Feature Engineering:**
   - Automated profiling evaluates features as they are.
   - It will never synthesize new domain representations, such as:
     - Extracting title (`"Master"`, `"Mrs"`, `"Mr"`) from `Name`
     - Combining `SibSp` + `Parch` + 1 into `FamilySize`
     - Binning `Age` into developmental life stages (`Child`, `Adult`, `Senior`)

4. **Blindness to Data Leakage:**
   - If a feature containing target leakage is present (e.g., `LifeboatNumber` predicting `Survived`), the profiler will praise it as a strong predictor, while a human engineer recognizes it as a fatal pipeline bug.

> **Best Practice Recommendation:**
> Run `ydata-profiling` in your first 5 minutes to audit dataset health and catch errors.
> Then spend your time doing thoughtful, hypothesis-driven manual EDA in notebooks.


### Key Takeaways
1. `ProfileReport` compiles comprehensive distributions, stats, and warnings into an interactive HTML report in seconds.
2. Use the **Alerts** section to identify high cardinality, zero-value counts, and missingness before building ML models.
3. Review non-linear correlation metrics like **Phik** to evaluate categorical feature associations.
4. Treat automated profiling as an initial diagnostic tool—never a substitute for domain understanding, hypothesis testing, and creative feature engineering.
